# Actividad 1 · Subsecuencia común más larga (LCS)
**Diseño Avanzado de Algoritmos** · Antonio Gil Alcaraz

Tres algoritmos para calcular la LCS de dos cadenas y su comparación en tiempo de CPU y memoria:

* **Versión A** – recursiva directa, sin memorización (Tema 1).
* **Versión B** – divide y conquista de Hirschberg (Tema 2).
* **Versión C** – programación dinámica tabular (Tema 3).

Repositorio: https://github.com/spyred/Actividad_1_D.A.A

**Cómo usar este cuaderno.** Ábrelo con Jupyter Lab desde la carpeta del proyecto (la que contiene
`lcs/`, `tests/`, `bench/` y `resultados/`) y ejecuta **Run ▸ Run All Cells**. Por defecto muestra los
resultados ya medidos en `resultados/`, que son los de la memoria; para repetir las mediciones, pon
`EJECUTAR_EXPERIMENTOS = True` en la sección 0 (tarda unos 5 minutos).

### Índice
0. Preparación del entorno
1. El problema de la LCS
2. Versión A · recursiva directa
3. Versión B · Hirschberg (divide y conquista)
4. Versión C · programación dinámica tabular
5. Juego de pruebas
6. Experimentos: tiempo de CPU y memoria
7. Análisis de los resultados
8. Discusión y conclusiones

## 0. Preparación del entorno

In [ ]:
# False -> usa los resultados ya guardados en resultados/ (los de la memoria)
# True  -> repite todas las mediciones (unos 5 minutos) y SOBRESCRIBE resultados/.
#          Las cifras nuevas ya no coincidirán exactamente con las de la memoria en PDF.
EJECUTAR_EXPERIMENTOS = False

In [ ]:
import csv
import json
import math
import platform
import subprocess
import sys
import unittest
from pathlib import Path

from IPython.display import Image, Markdown, display

RAIZ = Path.cwd()
assert (RAIZ / "lcs").is_dir() and (RAIZ / "tests").is_dir(), (
    "Abre el cuaderno desde la carpeta del proyecto (la que contiene lcs/, tests/ y bench/)")
RES = RAIZ / "resultados"

try:
    import matplotlib.pyplot as plt
    HAY_MATPLOTLIB = True
except ImportError:
    HAY_MATPLOTLIB = False

print("Python    ", platform.python_version())
print("Sistema   ", platform.system(), platform.release())
print("matplotlib", "sí" if HAY_MATPLOTLIB else "no (se mostrarán las figuras guardadas en resultados/)")

## 1. El problema de la LCS

Una **subsecuencia** de una cadena se obtiene borrando algunos símbolos sin cambiar el orden de los
demás. Dadas $x = x_1 \dots x_m$ e $y = y_1 \dots y_n$, buscamos una subsecuencia **común** de
longitud **máxima**. En `ABCBDAB` y `BDCABA`, tanto `BCBA` como `BDAB` son LCS de longitud 4: puede
haber varias soluciones óptimas.

Sea $L(i,j)$ la longitud de la LCS de los prefijos $x_1 \dots x_i$ e $y_1 \dots y_j$. Mirando el
último símbolo de cada prefijo (principio de optimalidad, Tema 3):

$$
L(i,j) =
\begin{cases}
0 & \text{si } i = 0 \text{ o } j = 0 \\[3pt]
L(i-1,\, j-1) + 1 & \text{si } x_i = y_j \\[3pt]
\max\big(L(i-1,\, j),\ L(i,\, j-1)\big) & \text{si } x_i \ne y_j
\end{cases}
$$

El problema tiene **subestructura óptima** y **subproblemas solapados**: sólo hay $(m+1)(n+1)$ pares
$(i,j)$ distintos, pero una recursión ingenua los visita muchas veces. Las tres versiones parten de
esta recurrencia; cambia **cómo** la evalúan.

## 2. Versión A · recursiva directa

Aplica la recurrencia tal cual, sin guardar resultados (`lcs/recursiva.py`).

In [ ]:
import sys

# Ampliamos el límite de recursión para cadenas largas en el mejor caso
sys.setrecursionlimit(max(10000, sys.getrecursionlimit()))


def lcs_recursiva(x, y, contador=None):
    """Devuelve una LCS de x e y. Si se pasa contador=[0], cuenta las llamadas."""

    def rec(i, j):
        if contador is not None:
            contador[0] += 1
        if i == 0 or j == 0:
            return ""
        if x[i - 1] == y[j - 1]:
            return rec(i - 1, j - 1) + x[i - 1]
        izq = rec(i - 1, j)
        der = rec(i, j - 1)
        if len(izq) >= len(der):
            return izq
        return der

    return rec(len(x), len(y))


c = [0]
print(lcs_recursiva("ABCBDAB", "BDCABA", c), "·", c[0], "llamadas")

### 2.1. Ecuación de recurrencia del coste (Tema 1)

* **Mejor caso** (cadenas iguales): siempre se toma la diagonal,
  $T(m,n) = T(m-1, n-1) + c$, luego $T = \Theta(\min(m,n))$.
* **Peor caso** (ningún símbolo en común): siempre se abren las dos ramas,
  $T(m,n) = T(m-1,n) + T(m, n-1) + c$. Con $k = m + n$, $T(k) \le 2T(k-1) + c$; la ecuación
  característica $(x-2)(x-1)=0$ da $T(k) = c_1 2^k + c_2 = O(2^{m+n})$.

El número exacto de llamadas en el peor caso es $N(m,n) = 2\binom{m+n}{n} - 1$, que con $m = n$ es
$\Theta(4^n/\sqrt{n})$: como en $T(n) = 3T(n-1)+1 = \Theta(3^n)$ de la Actividad 3, el árbol de
recursión multiplica sus nodos en cada nivel. Lo comprobamos:

In [ ]:
print(f"{'m=n':>4} {'disjuntas':>12} {'2·C(2n,n)-1':>13} {'iguales':>9}")
for n in range(1, 11):
    c1, c2 = [0], [0]
    lcs_recursiva("A" * n, "C" * n, c1)
    lcs_recursiva("A" * n, "A" * n, c2)
    print(f"{n:>4} {c1[0]:>12,} {2 * math.comb(2 * n, n) - 1:>13,} {c2[0]:>9}")

La fórmula coincide exactamente. La memoria de A es la pila de recursión, de profundidad
$\le m + n$: $\Theta(m+n)$.

## 3. Versión B · Hirschberg (divide y conquista)

Rellenar la tabla por filas guardando sólo la anterior da la *longitud* en espacio $\Theta(n)$, pero
no permite reconstruir la subsecuencia. Hirschberg la recupera dividiendo el problema
(`lcs/hirschberg.py`):

1. **Dividir.** $mid = \lfloor m/2 \rfloor$. $L_1$ = última fila de la tabla de $x_{1..mid}$ frente a
   $y$; $L_2$ = última fila de la tabla de $x_{mid+1..m}$ invertida frente a $y$ invertida. Con
   $k = \arg\max_j \, L_1[j] + L_2[n-j]$, una LCS óptima pasa por la columna $k$.
2. **Conquistar** $(x_{1..mid},\ y_{1..k})$ y $(x_{mid+1..m},\ y_{k+1..n})$ y **combinar** concatenando.

Casos base: alguna cadena vacía, o $x$ de un solo símbolo.

In [1]:
def ultima_fila(x, y):
    """Calcula la última fila de la tabla LCS de x frente a y usando O(len(y)) memoria."""
    n = len(y)
    prev = [0] * (n + 1)
    for xi in x:
        cur = [0] * (n + 1)
        for j in range(1, n + 1):
            if xi == y[j - 1]:
                cur[j] = prev[j - 1] + 1
            else:
                if prev[j] >= cur[j - 1]:
                    cur[j] = prev[j]
                else:
                    cur[j] = cur[j - 1]
        prev = cur
    return prev


def hirschberg_rec(x, y, partes):
    m = len(x)
    n = len(y)
    if m == 0 or n == 0:
        return
    if m == 1:
        if x in y:
            partes.append(x)
        return

    mid = m // 2
    l1 = ultima_fila(x[:mid], y)
    l2 = ultima_fila(x[mid:][::-1], y[::-1])

    mejor = -1
    k = 0
    for j in range(n + 1):
        suma = l1[j] + l2[n - j]
        if suma > mejor:
            mejor = suma
            k = j

    hirschberg_rec(x[:mid], y[:k], partes)
    hirschberg_rec(x[mid:], y[k:], partes)


def lcs_hirschberg(x, y):
    """Devuelve una LCS de x e y en espacio lineal O(m + n)."""
    partes = []
    hirschberg_rec(x, y, partes)
    return "".join(partes)


print(lcs_hirschberg("ABCBDAB", "BDCABA"))

BDAB


### 3.1. Recurrencia del coste (Tema 2)

$$T(m,n) = T\!\left(\tfrac{m}{2},\, k\right) + T\!\left(\tfrac{m}{2},\, n-k\right) + c\,mn$$

El Teorema Maestro no se aplica tal cual (dos variables y reparto desigual de $y$), pero el **árbol de
recursión** lo resuelve, como en la Actividad 2: en el nivel $i$ los subproblemas tienen $m/2^i$ filas
y sus columnas suman $n$, así que el nivel cuesta $c\,mn/2^i$ y

$$T(m,n) \le c\,mn\left(1 + \tfrac12 + \tfrac14 + \cdots\right) = 2c\,mn = \Theta(mn).$$

Hay $\lceil \log_2 m \rceil$ niveles y en cada uno sólo viven dos filas: memoria $\Theta(m+n)$. Se paga
hasta el doble de operaciones que C a cambio de memoria lineal.

## 4. Versión C · programación dinámica tabular

Se construye la tabla completa $L[0..m][0..n]$ por filas, como en el Entrenamiento 3 del Tema 3 y en
el Lab 1 de la mochila. $L[m][n]$ es la longitud; la subsecuencia se reconstruye desde $(m,n)$: si
$x_i = y_j$ se anota el símbolo y se va en diagonal; si no, se va a la celda vecina de mayor valor; al
final se invierte la lista (`lcs/dinamica.py`).

In [ ]:
def tabla_lcs(x, y):
    m = len(x)
    n = len(y)
    L = [[0] * (n + 1)]
    for i in range(1, m + 1):
        xi = x[i - 1]
        prev = L[i - 1]
        cur = [0] * (n + 1)
        for j in range(1, n + 1):
            if xi == y[j - 1]:
                cur[j] = prev[j - 1] + 1
            else:
                if prev[j] >= cur[j - 1]:
                    cur[j] = prev[j]
                else:
                    cur[j] = cur[j - 1]
        L.append(cur)
    return L


def reconstruir(L, x, y):
    i = len(x)
    j = len(y)
    sol = []
    while i > 0 and j > 0:
        if x[i - 1] == y[j - 1]:
            sol.append(x[i - 1])
            i -= 1
            j -= 1
        elif L[i - 1][j] >= L[i][j - 1]:
            i -= 1
        else:
            j -= 1
    sol.reverse()
    return "".join(sol)


def lcs_dinamica(x, y):
    """Devuelve una LCS de x e y construyendo la tabla completa O(m * n)."""
    L = tabla_lcs(x, y)
    return reconstruir(L, x, y)


print(lcs_dinamica("ABCBDAB", "BDCABA"))

La tabla del ejemplo, con las celdas de la reconstrucción marcadas con `*` (coincidencias que entran en
la LCS):

In [ ]:
def mostrar_tabla(x, y):
    L = tabla_lcs(x, y)
    marcadas = set()
    i, j = len(x), len(y)
    while i > 0 and j > 0:
        if x[i - 1] == y[j - 1]:
            marcadas.add((i, j))
            i -= 1
            j -= 1
        elif L[i - 1][j] >= L[i][j - 1]:
            i -= 1
        else:
            j -= 1
    print("     " + " ".join(f"{c:<2}" for c in ["ε"] + list(y)))
    for i, fila in enumerate(L):
        etiqueta = "ε" if i == 0 else x[i - 1]
        celdas = [f"{v}*" if (i, j) in marcadas else f"{v} " for j, v in enumerate(fila)]
        print(f"  {etiqueta}  " + " ".join(celdas))
    print(f"\nL[{len(x)}][{len(y)}] = {L[-1][-1]}  ·  LCS = {reconstruir(L, x, y)}")


mostrar_tabla("ABCBDAB", "BDCABA")

**Coste:** $(m+1)(n+1)$ celdas de coste constante, $\Theta(mn)$ en tiempo **y en memoria**; la
reconstrucción da como mucho $m+n$ pasos.

## 5. Juego de pruebas

Se ejecuta el juego de pruebas del proyecto (`tests/test_lcs.py`), sobre los módulos de `lcs/`:
casos límite, el ejemplo del Tema 3 (LCS de longitud 20), 300 parejas aleatorias comparadas con un
oráculo de fuerza bruta, igualdad de B y C en cadenas largas y la fórmula de llamadas de A.

In [ ]:
pruebas = unittest.TestLoader().discover(str(RAIZ / "tests"))
resultado = unittest.TextTestRunner(verbosity=2, stream=sys.stdout).run(pruebas)
print("✔ Todas las pruebas pasan" if resultado.wasSuccessful() else "✘ Hay pruebas que fallan")

In [ ]:
# Las versiones de este cuaderno dan el mismo resultado que las del proyecto
import lcs

X3, Y3 = "ACCGGTCGAGTGCGCGGAAGCCGGCCGAA", "GTCGTTCGGAATGCCGTTGCTCTGTAAA"
assert lcs_hirschberg(X3, Y3) == lcs.lcs_hirschberg(X3, Y3)
assert lcs_dinamica(X3, Y3) == lcs.lcs_dinamica(X3, Y3)
assert lcs_recursiva(X3[:12], Y3[:12]) == lcs.lcs_recursiva(X3[:12], Y3[:12])
print("Tema 3, Entrenamiento 3 ->", lcs_dinamica(X3, Y3), f"(longitud {len(lcs_dinamica(X3, Y3))})")

## 6. Experimentos: tiempo de CPU y memoria

Las mediciones las hacen los scripts del proyecto (`bench/`), igual que con `python ejecutar.py`:

* **Experimento 1:** seis parejas de ADN aleatorio (A, C, G, T; semilla fija) de 100, 250, 500, 1000,
  2000 y 4000 símbolos. Cada medición se lanza en un **proceso nuevo**.
* **Experimento 2:** A, B y C con cadenas cortas **iguales**, **aleatorias** y **disjuntas**, que fijan
  el mejor caso, el típico y el peor de A.
* **Tiempo de CPU** con `time.process_time()`, repitiendo cada cálculo hasta acumular al menos 0,25 s
  (en Windows ese reloj avanza a saltos de unos 15,6 ms) y tomando la mediana de 3 repeticiones.
* **Memoria máxima** con `tracemalloc` (lo que reserva el algoritmo), en una ejecución aparte; el RSS
  del proceso sirve de control.
* La versión A se corta a los 30 s; si agota el límite, no se lanza en tallas mayores.

In [ ]:
if EJECUTAR_EXPERIMENTOS:
    for script in ("bench/experimentos.py", "bench/graficas.py"):
        print(f"--- {script}")
        subprocess.run([sys.executable, script], cwd=RAIZ, check=True)
else:
    print("Se usan los resultados guardados en resultados/ (EJECUTAR_EXPERIMENTOS = False).")


def leer_csv(nombre):
    with open(RES / nombre, encoding="utf-8") as fh:
        return list(csv.DictReader(fh))


grande = leer_csv("experimento_grande.csv")
pequeno = leer_csv("experimento_pequeno.csv")
entorno = json.loads((RES / "entorno.json").read_text(encoding="utf-8"))
print(f"Mediciones hechas con Python {entorno['python']} en {entorno['sistema']}, {entorno['procesador']}")

### 6.1. Tablas de resultados

In [ ]:
def num(v, d=3):
    return f"{v:.{d}f}".replace(".", ",")


por_n = {}
for r in grande:
    por_n.setdefault(int(r["n"]), {})[r["version"]] = r

filas = ["| n | \\|LCS\\| | A | CPU B (s) | CPU C (s) | B / C | Pico B (MiB) | Pico C (MiB) | RSS C (MiB) |",
         "|--:|--:|:-:|--:|--:|--:|--:|--:|--:|"]
for n, d in sorted(por_n.items()):
    a, b, c = d["A"], d["B"], d["C"]
    tb, tc = float(b["cpu_mediana_s"]), float(c["cpu_mediana_s"])
    estado_a = num(float(a["cpu_mediana_s"])) + " s" if a["estado"] == "ok" else a["estado"]
    filas.append(f"| {n} | {c['len_lcs']} | {estado_a} | {num(tb)} | {num(tc)} | {num(tb / tc, 2)} | "
                 f"{num(int(b['pico_tracemalloc_bytes']) / 2**20, 2)} | "
                 f"{num(int(c['pico_tracemalloc_bytes']) / 2**20, 2)} | "
                 f"{num(int(c['rss_incremento_kib']) / 1024, 1)} |")
display(Markdown("**Tabla 1.** Experimento 1: tiempo de CPU (mediana de 3) y memoria.\n\n" + "\n".join(filas)))

filas = ["| Régimen | n | Llamadas de A | CPU A (ms) | CPU B (ms) | CPU C (ms) |",
         "|:--|--:|--:|--:|--:|--:|"]
for regimen in ("iguales", "aleatorias", "disjuntas"):
    r = max((f for f in pequeno if f["regimen"] == regimen), key=lambda f: int(f["n"]))
    llamadas = f"{int(r['llamadas_A']):,}".replace(",", ".")
    filas.append(f"| {regimen} | {r['n']} | {llamadas} | {num(float(r['cpu_A_s']) * 1e3)} | "
                 f"{num(float(r['cpu_B_s']) * 1e3)} | {num(float(r['cpu_C_s']) * 1e3)} |")
display(Markdown("**Tabla 2.** Experimento 2: mayor talla medida de A en cada régimen.\n\n" + "\n".join(filas)))

### 6.2. Gráficas

In [ ]:
COLOR = {"A": "#2a78d6", "B": "#eb6834", "C": "#1baf7a"}
NOMBRE = {"A": "A · recursiva directa", "B": "B · Hirschberg", "C": "C · PD tabular"}
ok = {v: [r for r in grande if r["version"] == v and r["estado"] == "ok"] for v in "ABC"}
ale = [f for f in pequeno if f["regimen"] == "aleatorias"]

if HAY_MATPLOTLIB:
    fig, (g1, g2) = plt.subplots(1, 2, figsize=(13, 4.3))
    for v in "BC":
        xs = [int(f["n"]) for f in ale] + [int(r["n"]) for r in ok[v]]
        ys = [float(f[f"cpu_{v}_s"]) for f in ale] + [float(r["cpu_mediana_s"]) for r in ok[v]]
        g1.plot(xs, ys, "o-", ms=4, color=COLOR[v], label=NOMBRE[v])
        g2.plot([int(r["n"]) for r in ok[v]], [int(r["pico_tracemalloc_bytes"]) / 2**20 for r in ok[v]],
                "o-", ms=4, color=COLOR[v], label=NOMBRE[v])
    g1.plot([int(f["n"]) for f in ale], [float(f["cpu_A_s"]) for f in ale], "o-", ms=4,
            color=COLOR["A"], label=NOMBRE["A"])
    cp = [f for f in ale if f.get("pico_A_bytes")]
    g2.plot([int(f["n"]) for f in cp], [int(f["pico_A_bytes"]) / 2**20 for f in cp], "o-", ms=4,
            color=COLOR["A"], label=NOMBRE["A"])
    g1.set(xscale="log", yscale="log", xlabel="longitud de cada cadena n", ylabel="tiempo de CPU (s)",
           title="Tiempo de CPU (cadenas aleatorias)")
    g2.set(xscale="log", yscale="log", xlabel="longitud de cada cadena n", ylabel="pico de memoria (MiB)",
           title="Memoria máxima reservada (tracemalloc)")
    g1.legend()
    g2.legend()
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 3.8))
    for regimen, (col, mk) in {"iguales": ("#008300", "s"), "aleatorias": (COLOR["A"], "o"),
                               "disjuntas": ("#6250d6", "^")}.items():
        fs = [f for f in pequeno if f["regimen"] == regimen]
        ax.plot([int(f["n"]) for f in fs], [int(f["llamadas_A"]) for f in fs], mk, color=col, label=regimen)
    n_dis = max(int(f["n"]) for f in pequeno if f["regimen"] == "disjuntas")
    ax.plot(range(1, 31), [k + 1 for k in range(1, 31)], "-", color="#52514e", lw=1, label="n + 1")
    ax.plot(range(1, n_dis + 1), [2 * math.comb(2 * k, k) - 1 for k in range(1, n_dis + 1)], "--",
            color="#52514e", lw=1, label="2·C(2n, n) − 1")
    ax.set(yscale="log", xlabel="n (m = n)", ylabel="llamadas de A", title="Versión A: llamadas recursivas")
    ax.legend(loc="center right")
    plt.show()
else:
    for nombre in ("fig_tiempo.png", "fig_memoria.png", "fig_llamadas_A.png"):
        if (RES / nombre).exists():
            display(Image(filename=str(RES / nombre)))

## 7. Análisis de los resultados

Si $T(n) \approx c \cdot n^p$, en escala log-log se ve una recta de pendiente $p$; la ajustamos por
mínimos cuadrados con $n \ge 500$. Para A, como el crecimiento es exponencial, ajustamos
$\log(\text{llamadas}) \approx a + b\,n$ en el régimen aleatorio y extrapolamos su tiempo.

In [ ]:
def recta(xs, ys):
    mx, my = sum(xs) / len(xs), sum(ys) / len(ys)
    b = sum((x - mx) * (y - my) for x, y in zip(xs, ys)) / sum((x - mx) ** 2 for x in xs)
    return my - b * mx, b


def pendiente(v, campo):
    rs = [r for r in ok[v] if int(r["n"]) >= 500]
    return recta([math.log(int(r["n"])) for r in rs], [math.log(float(r[campo])) for r in rs])[1]


for v in "BC":
    print(f"Versión {v}: tiempo ~ n^{pendiente(v, 'cpu_mediana_s'):.2f}    "
          f"memoria ~ n^{pendiente(v, 'pico_tracemalloc_bytes'):.2f}")

aj = [f for f in ale if int(f["n"]) >= 6]
a, b = recta([int(f["n"]) for f in aj], [math.log(int(f["llamadas_A"])) for f in aj])
medibles = [f for f in pequeno if float(f["cpu_A_s"]) > 0.05]
s_llamada = sum(float(f["cpu_A_s"]) for f in medibles) / sum(int(f["llamadas_A"]) for f in medibles)
print(f"\nVersión A (aleatorias): llamadas ~ {math.exp(b):.2f}^n, {s_llamada * 1e6:.2f} µs por llamada")
for n in (30, 50, 100, 1000):
    print(f"   n = {n:4d}: unos 10^{(a + b * n) / math.log(10) + math.log10(s_llamada):.0f} s")
print("   (la edad del universo es del orden de 4·10^17 s)")

nmax = max(por_n)
b_, c_ = por_n[nmax]["B"], por_n[nmax]["C"]
print(f"\nCon n = {nmax}: C usa {int(c_['pico_tracemalloc_bytes']) / 2**20:.1f} MiB y "
      f"B {int(b_['pico_tracemalloc_bytes']) / 2**20:.2f} MiB "
      f"-> {int(c_['pico_tracemalloc_bytes']) / int(b_['pico_tracemalloc_bytes']):.0f} veces menos memoria")
print(f"Con n = {nmax}: B tarda {float(b_['cpu_mediana_s']) / float(c_['cpu_mediana_s']):.2f} veces lo que C")

## 8. Discusión y conclusiones

**El coste exponencial de A procede de los subproblemas solapados.** C calcula cada $L(i,j)$ una sola
vez, $(n+1)^2$ celdas; A vuelve a calcular los mismos pares muchísimas veces y su número de llamadas
crece exponencialmente (Experimento 2 y sección 7). Además, su coste depende de los datos: con
cadenas **iguales** hace sólo $n+1$ llamadas y es la más rápida (Tabla 2), pero en cuanto hay
discrepancias se dispara. Probar sólo con entradas favorables ocultaría su peor caso.

**B y C: mismo orden de tiempo, distinto orden de memoria.** Las dos son $\Theta(mn)$ en tiempo, con
pendientes próximas a 2; B es algo más lenta porque recalcula hasta $2mn$ celdas y copia e invierte
trozos de cadena. En memoria la diferencia es de orden: C guarda la tabla completa, $\Theta(mn)$, y B
sólo dos filas, $\Theta(m+n)$; con 4000 símbolos, cientos de veces menos. Las pendientes algo
superiores a 2 se deben a que Python sólo reutiliza los enteros hasta 256: a partir de ahí cada valor
nuevo de la tabla ocupa un objeto propio.

**En la práctica:** C cuando la tabla cabe holgadamente en memoria (más rápida y sencilla); B cuando la
memoria es el recurso escaso; A sólo tiene interés didáctico o para cadenas minúsculas o casi idénticas.

**Limitaciones.** Python eleva las constantes, pero no los órdenes de crecimiento. `tracemalloc` no
cuenta los marcos de pila, así que infraestima la memoria de A. El tiempo de A varía mucho entre
parejas aleatorias de la misma talla, así que su extrapolación da órdenes de magnitud.

**Conclusiones**
1. Las ecuaciones de recurrencia predicen bien los tres comportamientos: A es exponencial en el peor
   caso (fórmula de llamadas verificada) y lineal en el mejor; B y C son $\Theta(mn)$ en tiempo; C usa
   memoria $\Theta(mn)$ y B, $\Theta(m+n)$.
2. La recursión directa es inviable más allá de unas decenas de símbolos.
3. La programación dinámica elimina la explosión exponencial resolviendo cada subproblema una sola vez,
   al precio de una tabla cuadrática.
4. Hirschberg reduce esa memoria cientos de veces a cambio de algo más de tiempo: conviene en cadenas largas.

### Referencias
1. Tema 1: Análisis de algoritmos recursivos.
2. Tema 2: Divide y conquista.
3. Tema 3: Programación dinámica.
4. Actividades 1–3 y Lab 1 (mochila 0/1).